<a href="https://colab.research.google.com/github/arildbn/bban4040/blob/main/martra-notebooks/3-2_openai-moderation-chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div>
    <h1>Large Language Models Projects</a></h1>
    <h3>Apply and Implement Strategies for Large Language Models</h3>
    <h2>3.2-Create a Moderation System using LangChain.</h2>
    <h3>OpenAI version</h3>
    <p>by <b>Pere Martra</b></p>
</div>

# How To Create a Moderation System Using LangChain.

We are goin to create a Moderation System based in two Models. The first Model  reads the User comments and answer them.

The second language Model receives the answer of the first model and identify any kind on negativity modifying if necessary the comment.

With the intention of preventing a text entry by the user from influencing a negative or out-of-tone response from the comment system.

In [ ]:
# === Colab dependency guard (bban4040) ===
# The langchain / langsmith stack upgrades transitive packages (requests,
# opentelemetry-*) past the exact versions Colab's preinstalled google
# packages pin (google-colab, google-adk, the otlp/gcp exporters), which
# prints noisy "pip's dependency resolver ... is incompatible" errors.
# We pin those families to the versions already installed so the installs
# below leave them untouched. PIP_CONSTRAINT is honored by every %pip call
# in this kernel. Harmless off Colab (nothing matches / nothing to pin).
import os, tempfile
from importlib import metadata

_keep = []
for _dist in metadata.distributions():
    _name = (_dist.metadata.get("Name") or "").strip()
    if not _name:
        continue
    if _name.lower() == "requests" or _name.lower().startswith("opentelemetry"):
        _keep.append(f"{_name}=={_dist.version}")

if _keep:
    _con = os.path.join(tempfile.gettempdir(), "bban4040_pip_constraints.txt")
    with open(_con, "w") as _f:
        _f.write("\n".join(sorted(set(_keep))) + "\n")
    os.environ["PIP_CONSTRAINT"] = _con
    print(f"Pinned {len(_keep)} package(s) to avoid Colab pip resolver conflicts.")


In [ ]:
#Install de LangChain and openai libraries.
%pip install -q langchain
%pip install -q langchain-openai


In [ ]:
# === portable-setup (bban4040) ===
# Secrets resolve from Colab "Secrets" (userdata) on Colab, or environment
# variables / a local .env file when running locally. Nothing is hardcoded.
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass
    value = os.environ.get(name, default)
    return value.strip() if isinstance(value, str) else value


_key = get_secret("OPENAI_API_KEY")
if _key:
    os.environ["OPENAI_API_KEY"] = _key


## Importing LangChain Libraries.
* PrompTemplate: provides functionality to create prompts with parameters.
* OpenAI:  To interact with the OpenAI models.

In [ ]:
#PrompTemplate is a custom class that provides funcrionality to create prompts
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [ ]:
import torch
import os
import numpy as np

We need an OpenAI key to interac with the OpenAI API.

Here you can access to your keys.
https://platform.openai.com/account/api-keys

OpenAI it's a pay service and you need a credit card to get a Key. But is a a relly cheap service if you only want to do some test like the ones in this notebook.

I'm using the gpt-3.5 as a moderator.


In [ ]:
from getpass import getpass
os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY") or getpass("OpenAI API Key: ")


In [ ]:
assistant_llm = ChatOpenAI(model="gpt-3.5-turbo")

Create the template for the first model called **assistant**.

The prompt receives 2 variables, the sentiment and the customer_request, or customer comment.

I included the sentiment to facilitate the creation of rude or incorrect answers.

In [ ]:
# Instruction how the LLM must respond the comments,
assistant_template = """
You are {sentiment} assistant that responds to user comments,
using similar vocabulary than the user.
User:" {customer_request}"
Comment:
"""

In [ ]:
#Create the prompt template to use in the Chain for the first Model.
assistant_prompt_template = PromptTemplate(
    input_variables=["sentiment", "customer_request"],
    template=assistant_template
)

Now we create a First Chain. Just chaining the assistant_prompt_template and the model. The model will receive the prompt generated with the prompt_template.  

In [ ]:
output_parser = StrOutputParser()
assistant_chain = assistant_prompt_template | assistant_llm | output_parser


To execute the chain created it's necessary to call the **.run** method of the chain, and pass the variables necessaries.

In our case: *customer_request* and *sentiment*.

In [ ]:
#Support function to obtain a response to a user comment.
def create_dialog(customer_request, sentiment):
    #calling the .invoke method from the chain created Above.
    assistant_response = assistant_chain.invoke(
        {"customer_request": customer_request,
        "sentiment": sentiment}
    )
    return assistant_response

## Obtain answers from our first Model Unmoderated.

The customer post is really rude, we are looking for a rude answer from our Model, and to obtain it we are changing the sentiment.

In [ ]:
# This is the customer request, or customer comment in the forum moderated by the agent.
# feel free to modify it.
customer_request = """This product is a piece of shit. I feel like an Idiot!"""


In [ ]:
# Our assistant working in 'nice' mode.
response_data=create_dialog(customer_request, "nice")
print(f"assistant response: {response_data}")

The answer obtained is really polite. It dosn't need moderation.

In [ ]:
#Our assistant running in rude mode.
response_data = create_dialog(customer_request, "most rude")
print(f"assistant response: {response_data}")

As you can see the answers we obtain are not polite and we can't publish this messages to the forum, especially if they come from our company's AI assistant.

## Moderator
Let's create the second moderator. It will receive the message generated previously and rewrite it if necessary.


In [ ]:
#The moderator prompt template
moderator_template = """
You are the moderator of an online forum, you are strict and will not tolerate any negative comments.
You will receive a Original comment and if it is impolite you must transform in polite.
Try to mantain the meaning when possible,

If it it's polite, you will let it remain as is and repeat it word for word.
Original comment: {comment_to_moderate}
"""
# We use the PromptTemplate class to create an instance of our template that will use the prompt from above and store variables we will need to input when we make the prompt.
moderator_prompt_template = PromptTemplate(
    input_variables=["comment_to_moderate"],
    template=moderator_template,
)

In [ ]:
#I'm going to use a more advanced LLM
moderator_llm = ChatOpenAI(model="gpt-4")

In [ ]:
#We build the chain for the moderator.
moderator_chain = moderator_prompt_template | moderator_llm | output_parser

In [ ]:
# To run our chain we use the .run() command
moderator_data = moderator_chain.invoke({"comment_to_moderate": response_data})

In [ ]:
print(moderator_data)

Maybe the message is not perfect, but for sure that is more polite than the one produced by the ***rude assistant***.

## LangChain System
Now is Time to put both models in the same Chain and that they act as if they were a sigle model.

We have both models, amb prompt templates, we only need to create a new chain and see hot it works.

First we create two chain, one for each pair of prompt and model.


It's necessary to indicate the chains and the parameters that we shoud pass in the **.invoke** method.

In [ ]:
assistant_moderated_chain = (
    {"comment_to_moderate":assistant_chain}
    |moderator_chain
)

Lets use our Moderating System!

In [ ]:
# We can now run the chain.
from langchain_core.tracers import ConsoleCallbackHandler
assistant_moderated_chain.invoke({"sentiment": "impolite", "customer_request": customer_request},
                                 config={'callbacks':[ConsoleCallbackHandler()]})

Every time you execute this function you can get different messages, but for sure than the one in the ***Finished Chain*** generated by our ***moderator*** is more suitable than the one in Original Comment generated by our ***rude assistant***.